In [1]:
import numpy as np
import os
import torch
from torch.utils.data import DataLoader
from matplotlib import pyplot as plt
from accelerate import Accelerator
from torch_ema import ExponentialMovingAverage as EMA
from torchvision import transforms as tf
import torch.distributed as dist

from wavediffusion.model_unet import myUnet
from wavediffusion.model import Scaled, PredX0
from wavediffusion.wavedata import npyDataWndHist
from wavediffusion.diffusion import ScheduleLogLinear, ScheduleDDPM, samples, samples_onestep, masked_training_loop

from wavediffusion.waveutils import evaluate, sample_and_save

%load_ext autoreload
%autoreload 2

In [2]:
train_file_path = '/global/homes/j/jiarongw/scratch_folder/wave_data/train_global/'
train_file_names = [
    *( (f'waveparts_2010{i:02d}', f'forcing_2010{i:02d}') for i in range(1, 13) ),
    *( (f'waveparts_2011{i:02d}', f'forcing_2011{i:02d}') for i in range(1, 13) ),
    *( (f'waveparts_2012{i:02d}', f'forcing_2012{i:02d}') for i in range(1, 13) ),
    *( (f'waveparts_2013{i:02d}', f'forcing_2013{i:02d}') for i in range(1, 13) ),
    *( (f'waveparts_2014{i:02d}', f'forcing_2014{i:02d}') for i in range(1, 13) ),
    *( (f'waveparts_2015{i:02d}', f'forcing_2015{i:02d}') for i in range(1, 13) ),
    *( (f'waveparts_2016{i:02d}', f'forcing_2016{i:02d}') for i in range(1, 13) ),
    *( (f'waveparts_2017{i:02d}', f'forcing_2017{i:02d}') for i in range(1, 13) ),
    *( (f'waveparts_2018{i:02d}', f'forcing_2018{i:02d}') for i in range(1, 13) ),
    *( (f'waveparts_2019{i:02d}', f'forcing_2019{i:02d}') for i in range(1, 13) ),
    *( (f'waveparts_2020{i:02d}', f'forcing_2020{i:02d}') for i in range(1, 13) ),
    *( (f'waveparts_2021{i:02d}', f'forcing_2021{i:02d}') for i in range(1, 13) ),
    *( (f'waveparts_2022{i:02d}', f'forcing_2022{i:02d}') for i in range(1, 13) ),
]
train_file_list = [(os.path.join(train_file_path, f'{x}.npy'), 
                    os.path.join(train_file_path, f'{f}.npy')) for x, f in train_file_names]   
stats_file1 = os.path.join(train_file_path, 'stats.npz')
stats = np.load(stats_file1)
meanf, stdf = stats['meanf'], stats['stdf']
stats_file2 = os.path.join(train_file_path, 'stats_parts.npz')
stats_parts = np.load(stats_file2)
meanx, stdx = stats_parts['meanparts'], stats_parts['stdparts']
train = npyDataWndHist(
    train_file_list,
    resize_x=(320,320), resize_f=(320,320), 
    landmaskname=os.path.join(train_file_path, 'mask.npy'),
    use_icymask=True, compute_stats=False,
    meanx=meanx, stdx=stdx, meanf=meanf, stdf=stdf,
    OPTION=3
)

In [3]:
train_file_path = '/global/homes/j/jiarongw/scratch_folder/wave_data/train_global/'
test_file_path = '/global/homes/j/jiarongw/scratch_folder/wave_data/test_global/'
test_file_names = [*( (f'wave_2004{i:02d}', f'forcing_2004{i:02d}') for i in range(1, 13) )]
# test_file_names = [('wave_200409', 'forcing_200409')]
test_file_list = [(os.path.join(test_file_path, f'{x}.npy'), 
                   os.path.join(test_file_path, f'{f}.npy')) for x, f in test_file_names]
stats_file = os.path.join(train_file_path, 'stats.npz')
stats = np.load(stats_file)
meanx, stdx = stats['meanx'], stats['stdx']
meanf, stdf = stats['meanf'], stats['stdf']
test = npyDataWndHist(
    test_file_list,
    resize_x=(320,320), resize_f=(320,320), 
    landmaskname=os.path.join(test_file_path, 'mask.npy'),
    use_icymask=True, compute_stats=False,
    meanx=meanx, stdx=stdx, meanf=meanf, stdf=stdf
)

In [3]:
### Predicting epsilon
n_ensem = 20
epoch = 16
RESUME = True
path = '/global/homes/j/jiarongw/scratch_folder/log1p/hist1/'
weights_file = path + f'ckpt_{epoch}.pt'
model = Scaled(myUnet)(in_dim=320, in_ch=4, out_ch=4, ch=256, precond_ch=13, 
                       scale=(test.meanx, test.stdx, test.meanf, test.stdf),
                       ch_mult=(1, 2, 2), attn_resolutions=(16,))  

ema = EMA(model.parameters(), decay=0.999)
if RESUME:
    ckpt = torch.load(weights_file, map_location="cpu")
    model.load_state_dict(ckpt["model"])
    ema.load_state_dict(ckpt["ema"])
a = Accelerator()
model = a.prepare(model)
ema.to(a.device)

In [8]:
### Predicting state
n_ensem = 20
epoch = 8
RESUME = True
path = '/global/homes/j/jiarongw/scratch_folder/log1p/hist1_PredX0/'
weights_file = path + f'ckpt_{epoch}.pt'
model = PredX0(Scaled(myUnet))(in_dim=320, in_ch=4, out_ch=4, ch=256, precond_ch=13, 
                       scale=(test.meanx, test.stdx, test.meanf, test.stdf),
                       ch_mult=(1, 2, 2), attn_resolutions=(16,))  

ema = EMA(model.parameters(), decay=0.999)
if RESUME:
    ckpt = torch.load(weights_file, map_location="cpu")
    model.load_state_dict(ckpt["model"])
    ema.load_state_dict(ckpt["ema"])
a = Accelerator()
model = a.prepare(model)
ema.to(a.device)

In [9]:
# Given one data point, infer conditional mean (unnormalized) 
# Also return the unnormalized x_truth and forcing for evaluation
from wavediffusion.diffusion import samples_onestep
@torch.no_grad()
def sample_onestep (f, x, mask, sigma_max=20):
    global a, test, ema, model
    with ema.average_parameters():
        x0 = samples_onestep(model, sigma_max=sigma_max, batchsize=1, cond=f, 
                             accelerator=a, mask=mask)   
    x_pred = test.invert_x(x0[0])
    x_pred = x_pred * tf.Resize((320,720))(mask[0].to(x_pred))
    x_truth = test.invert_x(x[0]) * tf.Resize((320,720))(mask[0].to(x))
    x_truth = x_truth.cpu().numpy()
    f_ = test.invert_f(f[0]).cpu().numpy()
    return x_pred.cpu().numpy(), x_truth, f_

In [5]:
wlat = np.load('/global/homes/j/jiarongw/scratch_folder/wave_data/wlat.npy')
# weighted and masked RMSE
def weighted_mse(a, b, mask, w):
    diff2 = (a - b) ** 2
    return np.sum(diff2[mask] * w[mask]) / np.sum(w[mask])
# weighted and masked spread / skill
def weighted_meanvar(std, mask, w):
    var = std ** 2
    return np.sum(var[mask] * w[mask]) / np.sum(w[mask])  

In [10]:
from tqdm import tqdm
hs_mse, lp_mse, tp_mse, thetap_mse, spread_mse = [], [], [], [], []
x_truth, mean, std = None, None, None
for index in tqdm(range(0, test.__len__(), 80)):
    print(f'Infering conditional mean for index {index}...')
    x, f, icymask = test.__getitem__(index)
    x = x.unsqueeze(0); f = f.unsqueeze(0); icymask = icymask.unsqueeze(0)
    x_pred, x_truth, f_ = sample_onestep (f, x, icymask, sigma_max=120)
    # Lower bound the wave length with 0?
    x_pred[1] = np.maximum(x_pred[1], 0)
    x_truth[1] = np.maximum(x_truth[1], 0) 
    # Use the ice mask?
    icymask = f_[2] == 1
    hs_mse.append(weighted_mse(x_truth[0], x_pred[0], icymask, wlat))
    lp_mse.append(weighted_mse(x_truth[1], x_pred[1], icymask, wlat))
    tp_mse.append(weighted_mse((x_truth[1]/1.56)**0.5, (x_pred[1]/1.56)**0.5, icymask, wlat))
    thetap_mse.append(weighted_mse(x_truth[2], x_pred[2], icymask, wlat))
    spread_mse.append(weighted_mse(x_truth[3], x_pred[3], icymask, wlat))  

  0%|          | 0/36 [00:00<?, ?it/s]

Infering conditional mean for index 0...


  3%|▎         | 1/36 [00:00<00:13,  2.56it/s]

Infering conditional mean for index 80...


  6%|▌         | 2/36 [00:00<00:10,  3.23it/s]

...

 94%|█████████▍| 34/36 [00:09<00:00,  3.45it/s]

Infering conditional mean for index 2720...


 97%|█████████▋| 35/36 [00:09<00:00,  3.33it/s]

Infering conditional mean for index 2800...


100%|██████████| 36/36 [00:10<00:00,  3.52it/s]


In [11]:
hs_mse = np.array(hs_mse); lp_mse = np.array(lp_mse); tp_mse = np.array(tp_mse); thetap_mse = np.array(thetap_mse) ; spread_mse = np.array(spread_mse) 
print(f'hs mse: {hs_mse.mean()**0.5:.2f} \\pm {np.std(hs_mse**0.5):.2f}')
print(f'lp mse: {lp_mse.mean()**0.5:.2f} \\pm {np.std(lp_mse**0.5):.2f}')
print(f'tp mse: {tp_mse.mean()**0.5:.2f} \\pm {np.std(tp_mse**0.5):.2f}')
print(f'thetap mse: {thetap_mse.mean()**0.5:.2f} \\pm {np.std(thetap_mse**0.5):.2f}')
print(f'spread mse: {spread_mse.mean()**0.5:.2f} \\pm {np.std(spread_mse**0.5):.2f}')

hs mse: 0.20 \pm 0.02
lp mse: 19.91 \pm 3.51
tp mse: 0.64 \pm 0.09
thetap mse: 30.11 \pm 7.06
spread mse: 3.99 \pm 0.47


In [7]:
GUIDED = False
n_ensem = 10
from tqdm import tqdm
@torch.no_grad()
def sample (f, x, mask, xt=None, n_ensem=10):
    global a, test, ema, model, schedule_infer, GUIDED 
    f = f.repeat(n_ensem, 1, 1, 1)    
    xt = xt.repeat(n_ensem, 1, 1, 1) * mask.to(a.device) if xt is not None else None
    if xt is None:
        print("Start with no history snapshot.")
    x0_ensem = []
    with ema.average_parameters():
        if GUIDED:
            hs_thres = -test.meanx[0]/test.stdx[0]
            *xt, x0 = samples_thres(model, schedule_infer.sample_sigmas(40), gam=1, mu=0.5, batchsize=n_ensem, 
                                    thres=hs_thres, accelerator=a, cond=f, mask=mask, xt=xt)
        else:
            *xt, x0 = samples(model, schedule_infer.sample_sigmas(40), gam=1, mu=0.5, batchsize=n_ensem,
                              accelerator=a, cond=f, mask=mask, xt=xt)
        # This operation hasn't been broadcasted
        for i in range(0, n_ensem):
            x_ = test.invert_x(x0[i])
            x_ = x_ * tf.Resize((320,720))(mask[0].to(x_))
            x0_ensem.append(x_.cpu().numpy())
    
    x_truth = test.invert_x(x[0]) * tf.Resize((320,720))(mask[0].to(x))
    x_truth = x_truth.cpu().numpy()
    f_ = test.invert_f(f[0]).cpu().numpy()
    x0_ensem = np.array(x0_ensem)
    
    mean = x0_ensem.mean(axis=0)
    std = x0_ensem.std(axis=0)    
    
    return x_truth, mean, std, x0_ensem[0]

hs_mse, lp_mse, tp_mse, thetap_mse, spread_mse = [], [], [], [], []
hs_var, lp_var, thetap_var, spread_var = [], [], [], []   

schedule_infer = ScheduleLogLinear(sigma_min=0.01, sigma_max=80, N=80)
x_truth, mean, std = None, None, None
for index in tqdm(range(0, test.__len__(), 80)):
    print(f'Sampling for index {index}...')
    x, f, icymask = test.__getitem__(index)
    x = x.unsqueeze(0); f = f.unsqueeze(0); icymask = icymask.unsqueeze(0)
    x_truth, x_mean, std, rsample = sample (f, x, icymask, xt=None, n_ensem=n_ensem)
    f_ = test.invert_f(f[0]).cpu().numpy()
    # Lower bound the wave length with 0?
    x_mean[1] = np.maximum(x_mean[1], 0)
    x_truth[1] = np.maximum(x_truth[1], 0)     

    # Use the ice mask?
    icymask = f_[2] == 1
    hs_mse.append(weighted_mse(x_truth[0], x_mean[0], icymask, wlat))
    lp_mse.append(weighted_mse(x_truth[1], x_mean[1], icymask, wlat))
    tp_mse.append(weighted_mse((x_truth[1]/1.56)**0.5, (x_mean[1]/1.56)**0.5, icymask, wlat))
    thetap_mse.append(weighted_mse(x_truth[2], x_mean[2], icymask, wlat))
    spread_mse.append(weighted_mse(x_truth[3], x_mean[3], icymask, wlat))
    hs_var.append(weighted_meanvar(std[0], icymask, wlat))
    lp_var.append(weighted_meanvar(std[1], icymask, wlat))
    thetap_var.append(weighted_meanvar(std[2], icymask, wlat))
    spread_var.append(weighted_meanvar(std[3], icymask, wlat))  
      
hs_mse = np.array(hs_mse); lp_mse = np.array(lp_mse); tp_mse = np.array(tp_mse); thetap_mse = np.array(thetap_mse) ; spread_mse = np.array(spread_mse) 
hs_var = np.array(hs_var); lp_var = np.array(lp_var); thetap_var = np.array(thetap_var); spread_var = np.array(spread_var)

print(f'hs mse: {hs_mse.mean()**0.5:.2f} \\pm {np.std(hs_mse**0.5):.2f}')
print(f'lp mse: {lp_mse.mean()**0.5:.2f} \\pm {np.std(lp_mse**0.5):.2f}')
print(f'tp mse: {tp_mse.mean()**0.5:.2f} \\pm {np.std(tp_mse**0.5):.2f}')
print(f'thetap mse: {thetap_mse.mean()**0.5:.2f} \\pm {np.std(thetap_mse**0.5):.2f}')
print(f'spread mse: {spread_mse.mean()**0.5:.2f} \\pm {np.std(spread_mse**0.5):.2f}')

print(f'hs ssr: {(hs_var.mean() / hs_mse.mean())**0.5:.2f}')
print(f'lp ssr: {(lp_var.mean() / lp_mse.mean())**0.5:.2f}')
print(f'thetap ssr: {(thetap_var.mean() / thetap_mse.mean())**0.5:.2f}')
print(f'spread ssr: {(spread_var.mean() / spread_mse.mean())**0.5:.2f}') 

/global/u1/j/jiarongw/wavediffusion/src/wavediffusion/wavedata.py:397: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /mscratch/sd/s/swowner/pytorch-build/pytorch/2.6.0/pytorch/torch/csrc/utils/tensor_numpy.cpp:203.)
  f = torch.from_numpy(self.F_files[file_idx][local_idx]).float()


Sampling for index 0...
Start with no history snapshot.


  3%|▎         | 1/36 [00:25<15:05, 25.88s/it]

Sampling for index 80...
Start with no history snapshot.


  6%|▌         | 2/36 [00:51<14:34, 25.72s/it]

...

Sampling for index 2720...
Start with no history snapshot.


 97%|█████████▋| 35/36 [14:58<00:25, 25.66s/it]

Sampling for index 2800...
Start with no history snapshot.


100%|██████████| 36/36 [15:23<00:00, 25.66s/it]

hs mse: 0.74 \pm 0.29
lp mse: 92.46 \pm 23.35
tp mse: 2.14 \pm 0.42
thetap mse: 37.57 \pm 8.65
spread mse: 5.58 \pm 0.68
hs ssr: 0.06
lp ssr: 0.06
thetap ssr: 0.07
spread ssr: 0.07


In [7]:
@torch.no_grad()
def sanity_check():
    global model, ema, a
    index = 0
    x, f, icymask = test.__getitem__(index)
    x = x.unsqueeze(0).to(a.device); f = f.unsqueeze(0).to(a.device); icymask = icymask.unsqueeze(0).to(a.device)
    sigma0 = torch.tensor([20.]).to(a.device)
    # xt = torch.zeros((n_ensem,) + model.input_dims).to(a.device) * sigma0 * mask.to(a.device)
    eps = torch.zeros((1,) + model.input_dims).to(a.device) * icymask.to(a.device)
    with ema.average_parameters():
        loss_method1 = model.get_loss_masked(x, sigma0, eps, cond=f, mask=icymask)
        pred = model.forward(x + sigma0 * eps, sigma0, cond=f) * icymask 
        loss_map = (x*icymask - pred) ** 2
        loss_method2 = loss_map.mean()
        print(loss_method1)
        print(loss_method2)

sanity_check()

/global/u1/j/jiarongw/wavediffusion/src/wavediffusion/wavedata.py:383: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /mscratch/sd/s/swowner/pytorch-build/pytorch/2.6.0/pytorch/torch/csrc/utils/tensor_numpy.cpp:203.)
  f = torch.from_numpy(self.F_files[file_idx][local_idx]).float()


tensor(0.4888, device='cuda:0')
tensor(0.4886, device='cuda:0')


In [5]:
import inspect
print(inspect.getsource(a.unwrap_model(model).get_loss_masked))

    def get_loss_masked(self, x0, sigma, eps, cond=None, mask=None, loss=nn.MSELoss):
        if mask != None:
            return loss()(x0*mask, self(x0 + sigma * eps, sigma, cond=cond)*mask)
        else:
            print('Should provide mask!')



In [53]:
pred_raw = a.unwrap_model(model).forward(x + sigma0 * eps, sigma0, cond=f)
loss1 = torch.nn.functional.mse_loss(x * icymask, pred_raw * icymask)
loss2 = ((x * icymask - pred_raw * icymask) ** 2).mean()
print(loss1, loss2, (loss1 - loss2).abs())

tensor(0.4920, device='cuda:0', grad_fn=<MseLossBackward0>) tensor(0.4920, device='cuda:0', grad_fn=<MeanBackward0>) tensor(0., device='cuda:0', grad_fn=<AbsBackward0>)
